# Benchmark: Reranker & Consensus Config Sweep

A small, honest benchmark comparing 16 pipeline configurations:

| Config | Reranker | Generators | Query Exp Strg|
|--------|----------|------------|---------------|
| A      | ON       | 3          | Both          |
| B      | ON       | 3          | HyDE          | 
| C      | ON       | 3          | MultiQuery    |
| D      | ON       | 3          | OFF           | 
| E      | ON       | 1          | Both          |
| F      | ON       | 1          | HyDE          |
| G      | ON       | 1          | MultiQuery    |
| H      | ON       | 1          | OFF           |
| I      | OFF      | 3          | Both          |
| J      | OFF      | 3          | HyDE          |
| K      | OFF      | 3          | MultiQuery    |
| L      | OFF      | 3          | OFF           |
| M      | OFF      | 1          | Both          |
| N      | OFF      | 1          | HyDE          |
| O      | OFF      | 1          | MultiQuery    |
| P      | OFF      | 1          | OFF           |




**Metrics:** latency, confidence scores (claim support, final), hallucination risk, chunk counts, and an LLM-judge grounding score (1-5).

15 fixed queries over 5 topics: RAG basics, vector databases, LLM evaluation, cross-encoder reranking, multi-agent consensus.

In [1]:
%load_ext autoreload
%autoreload 2 
import os, sys, json, time, asyncio, textwrap, pprint, copy
from pathlib import Path
from dataclasses import dataclass, field
from typing import Optional
import matplotlib.pyplot as plt

REPO_ROOT = Path.home() / "Desktop" / "Multiagent_RAG_system"
os.chdir(str(REPO_ROOT))
sys.path.insert(0, str(REPO_ROOT))

In [2]:
#%load_ext autoreload
#%autoreload 2 
# Forcing local env for notebook by disabling network-dependent features we don't need
os.environ.setdefault("QDRANT_ENDPOINT", "")
os.environ.setdefault("OTEL_ENABLED", "false")
os.environ.setdefault("LANGCHAIN_TRACING_V2", "false")

# Disable cache so every query runs through the full pipeline
os.environ["REDIS_URL"] = ""

In [3]:
#%load_ext autoreload
#%autoreload 2 
from multiagent_rag_system.src.utils.config_loader import get_settings
get_settings.cache_clear()
#os.environ.setdefault("ACTIVE_PROVIDER", 'groq')
settings = get_settings()

print(f"App:            {settings.app_name} v{settings.app_version}")
print(f"LLM:            {settings.active_provider.value}")
print(f"Embedding:      {settings.embeddings.model}")
print(f"Query Expander: {settings.query_expansion.strategy}    (default enabled = {settings.query_expansion.enabled})")
print(f"Reranker:       {settings.reranker.model}  (default enabled={settings.reranker.enabled})")
print(f"Consensus:      {settings.agents.consensus_n_agents} generators")

App:            MultiAgentRAG v1.0.0
LLM:            gemini
Embedding:      sentence-transformers/all-MiniLM-L6-v2
Query Expander: ExpansionStrategy.BOTH    (default enabled = True)
Reranker:       cross-encoder/ms-marco-MiniLM-L-6-v2  (default enabled=True)
Consensus:      3 generators


---
## 1. Initialize Core Components

In [4]:
%load_ext autoreload
%autoreload 2 
from multiagent_rag_system.src.embedding.embedding import get_embedder
from multiagent_rag_system.src.database.vector_store import get_vector_store

async def init_components():
    #1
    t0 = time.perf_counter()
    embedder = await get_embedder()
    print(f"Embedder ready  ({time.perf_counter()-t0:.1f}s)")
    #2
    t0 = time.perf_counter()
    store = await get_vector_store()
    print(f"Vector store ready  ({time.perf_counter()-t0:.1f}s)")
    count = await store.count()
    print(f"  Collection: {store._collection_name!r}  points: {count}")
    return embedder, store

embedder, store = await init_components()

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


/home/emmanuel/Desktop/Multiagent_RAG_system/.venv/lib/python3.13/site-packages/requests/__init__.py:109: RequestsDependencyWarning: urllib3 (2.7.0) or chardet (7.4.3)/charset_normalizer (3.4.9) doesn't match a supported version!
  warnings.warn(
/home/emmanuel/Desktop/Multiagent_RAG_system/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
PyTorch version 2.11.0 available.
No device provided, using cpu
Loading SentenceTransformer model from sentence-transformers/all-MiniLM-L6-v2.
{"model": "sentence-transformers/all-MiniLM-L6-v2", "timestamp": "2026-08-06T01:55:01.311008Z", "level": "info", "event": "embedding_model_loaded"}


Embedder ready  (82.8s)


HTTP Request: GET https://3ebbc886-dbf1-4aab-97d1-e1c1bee47fa6.eu-west-2-0.aws.cloud.qdrant.io:6333/collections/rag_chunks/exists "HTTP/1.1 200 OK"
HTTP Request: POST https://3ebbc886-dbf1-4aab-97d1-e1c1bee47fa6.eu-west-2-0.aws.cloud.qdrant.io:6333/collections/rag_chunks/points/count "HTTP/1.1 200 OK"
{"timestamp": "2026-08-06T01:55:07.043388Z", "level": "info", "event": "Qdrant ready  collection='rag_chunks'points=22  dim=384"}


Vector store ready  (5.7s)


HTTP Request: POST https://3ebbc886-dbf1-4aab-97d1-e1c1bee47fa6.eu-west-2-0.aws.cloud.qdrant.io:6333/collections/rag_chunks/points/count "HTTP/1.1 200 OK"


  Collection: 'rag_chunks'  points: 22


---
## 2. Data Check — Ingest if Empty

Only ingests if the vector store is empty (no-op on subsequent runs).

In [6]:
from multiagent_rag_system.agent.agents.doc_ingestion import DocumentIngestionPipeline
from multiagent_rag_system.src.models.models import IngestRequest

SAMPLE_DOCS = {
    "rag_basics.txt": (
        "Retrieval-Augmented Generation (RAG) is a technique that combines "
        "information retrieval with text generation. A RAG system first retrieves "
        "relevant document chunks from a knowledge base using vector similarity search, "
        "then feeds them as context to a large language model. This grounding reduces "
        "hallucinations because the LLM must answer from the provided passages rather "
        "than relying solely on parametric memory. Modern RAG pipelines include query "
        "expansion to improve recall, cross-encoder reranking to refine relevance, and "
        "multi-agent consensus to boost answer quality. RAG is widely used in enterprise "
        "chatbots, document Q&A, and research paper analysis."
    ),
    "vector_databases.txt": (
        "Vector databases store embeddings — dense numerical representations of data — "
        "and enable fast approximate nearest neighbour (ANN) search. Qdrant is a "
        "high-performance vector database written in Rust. It supports HNSW (Hierarchical "
        "Navigable Small World) indexes for sub-linear ANN search, payload filtering, "
        "and horizontal scaling. Other popular vector databases include Pinecone, "
        "Weaviate, Milvus, and Chroma. All of them use algorithms like HNSW, IVF, "
        "or DiskANN to balance recall, latency, and memory usage. Cosine similarity "
        "and dot product are the most common distance metrics. Vector databases are "
        "a core component of production RAG systems."
    ),
    "llm_evaluation.txt": (
        "Evaluating LLM outputs is crucial for building trustworthy AI systems. "
        "RAGAS (Retrieval Augmented Generation Assessment) is a framework that "
        "measures faithfulness, answer relevancy, context precision, and context recall. "
        "Faithfulness checks whether the answer is factually grounded in the retrieved "
        "context. Answer relevancy measures how well the response addresses the query. "
        "Context precision evaluates whether the retrieved chunks contain useful information. "
        "These metrics correlate well with human judgment. A good RAG pipeline typically "
        "scores above 0.85 on faithfulness, though the threshold depends on the use case. "
        "Hallucination risk is usually assessed per claim using lexical overlap or "
        "LLM-based entailment checking."
    ),
    "multi_agent_consensus.txt": (
        "Multi-agent consensus improves answer reliability by generating multiple "
        "independent answers and selecting the best one. In a typical setup, 3-5 LLM "
        "instances each produce an answer from the same retrieved context, using slightly "
        "different temperatures to introduce diversity. A word-frequency majority vote "
        "then picks the answer whose vocabulary best aligns with the group. This approach "
        "reduces the risk of a single bad generation dominating the output. Claim "
        "verification further strengthens trust by checking each sentence in the final "
        "answer against the source passages using lexical overlap or LLM-based entailment."
    ),
    "query_expansion.txt": (
        "Query expansion improves recall in retrieval systems by reformulating the "
        "user's original query into multiple related variants before searching the "
        "vector index. Common techniques include synonym injection, LLM-based query "
        "rewriting, and HyDE (Hypothetical Document Embeddings), where the model first "
        "generates a hypothetical answer and embeds that instead of the raw query. "
        "Multi-query expansion issues several reformulated queries in parallel and "
        "merges the retrieved candidate sets, typically via reciprocal rank fusion. "
        "Over-expansion can hurt precision by pulling in loosely related chunks, so "
        "expansion is usually paired with a reranking stage downstream."
    ),
    "embedding_models.txt": (
        "Embedding models convert text into dense vectors that capture semantic meaning. "
        "Popular choices include OpenAI's text-embedding-3, Cohere's embed-v3, and open "
        "models like BAAI/bge-large and E5. Embedding dimensionality typically ranges from "
        "384 to 3072, with higher dimensions capturing more nuance at the cost of storage "
        "and search latency. Matryoshka representation learning allows a single embedding "
        "model to produce truncatable vectors that retain most of their quality at smaller "
        "sizes. Choice of embedding model significantly affects retrieval quality "
        "independent of the vector database or indexing algorithm used."
    ),
    "chunking_strategies.txt": (
        "Chunking strategy has a large effect on RAG retrieval quality. Fixed-size "
        "chunking splits text every N tokens, often with overlap to preserve context "
        "across boundaries. Semantic chunking instead splits at natural topic "
        "boundaries detected via embedding similarity between adjacent sentences. "
        "Recursive character splitting tries progressively smaller separators (paragraph, "
        "sentence, word) until chunks fit a target size. Document-aware chunking respects "
        "structural elements like headers, tables, and code blocks. Chunks that are too "
        "small lose context; chunks that are too large dilute the embedding and hurt "
        "retrieval precision."
    ),
    "cross_encoder_reranking.txt": (
        "Cross-encoder reranking is a two-stage retrieval strategy. The first stage uses "
        "a fast bi-encoder (e.g. sentence-transformers) to retrieve candidate passages. "
        "The second stage applies a slower but more accurate cross-encoder model that "
        "jointly processes the query and each candidate passage. Cross-encoders like "
        "MS MARCO MiniLM and jina-reranker-v3 significantly improve precision by capturing "
        "deep query-document interactions that bi-encoders miss. The reranker typically "
        "selects the top 3-5 passages from an initial set of 10-20 candidates. This "
        "two-stage approach balances latency and accuracy for production RAG deployments."
    ),
    "prompt_engineering.txt": (
        "Prompt engineering is the practice of designing inputs to elicit better outputs "
        "from a language model. Techniques include few-shot examples, chain-of-thought "
        "prompting, and explicit output format constraints such as JSON schemas. System "
        "prompts set persistent behavior, while user prompts carry task-specific instructions. "
        "In RAG contexts, prompt templates typically instruct the model to answer only "
        "from the provided context and to say so explicitly when the context is "
        "insufficient, which reduces hallucination independent of retrieval quality."
    ),
    "coffee_brewing.txt": (
        "Pour-over coffee brewing relies on controlling water temperature, grind size, "
        "and pour rate. Water between 90-96°C extracts optimal flavor without scorching "
        "the grounds. A medium-coarse grind suits most pour-over methods like the V60 or "
        "Chemex. The bloom phase — a small initial pour that lets the coffee degas — "
        "typically lasts 30-45 seconds before the main pour begins. Total brew time for "
        "a standard 250ml cup usually falls between 2.5 and 3.5 minutes."
    ),
    "urban_gardening.txt": (
        "Urban gardening in small spaces benefits from container selection based on root "
        "depth: shallow-rooted herbs like basil need only 15cm of soil, while tomatoes "
        "require at least 40cm. Self-watering containers reduce watering frequency by "
        "maintaining a reservoir that wicks moisture upward. Companion planting — such as "
        "pairing marigolds with vegetables — can reduce pest pressure without pesticides. "
        "Balcony gardens should account for wind exposure, which dries soil faster than "
        "ground-level gardens."
    ),
    "model_quantization.txt": (
        "Quantization reduces the numerical precision of model weights and activations "
        "to shrink memory footprint and speed up inference. INT8 quantization typically "
        "halves memory use relative to FP16 with minimal accuracy loss, while INT4 "
        "methods like GPTQ and AWQ push further by identifying and preserving salient "
        "weight channels. Quantization-aware training simulates low-precision arithmetic "
        "during training to recover accuracy that post-training quantization would lose. "
        "Quantization is unrelated to retrieval quality in RAG systems but is often "
        "applied to the LLM component to reduce serving cost."
    ),
}

count = await store.count()
if count == 0:
    pipeline = DocumentIngestionPipeline()
    for filename, content in SAMPLE_DOCS.items():
        req = IngestRequest(content=content, metadata={"source": filename})
        resp = await pipeline.ingest_text(req)
        print(f"  {filename:35s} → {resp.chunks_created} chunks")
    count = await store.count()
    print(f"\nIngested {len(SAMPLE_DOCS)} documents -> {count} total chunks")
else:
    print(f"Vector store already has {count} points —> skipping ingestion.")

HTTP Request: POST https://3ebbc886-dbf1-4aab-97d1-e1c1bee47fa6.eu-west-2-0.aws.cloud.qdrant.io:6333/collections/rag_chunks/points/count "HTTP/1.1 200 OK"


Vector store already has 22 points —> skipping ingestion.


---
## 3. Eval Set

25 fixed queries: 5 per document topic. These are designed to be answerable from the ingested content.

In [7]:

EVAL_QUERIES = [
        # rag_basics.txt
        "What is Retrieval-Augmented Generation?",
      "How does RAG reduce hallucinations in LLM outputs?",
     "What are the main components of a modern RAG pipeline?",
       "Why is RAG widely used in enterprise applications?",
        "What technique does RAG use to retrieve relevant information?",
        # vector_databases.txt
        "What is a vector database used for?",
        "How does Qdrant perform fast similarity search?",
        "What are some popular vector databases besides Qdrant?",
        "What distance metrics are commonly used in vector databases?",
        "What indexing algorithm does Qdrant use?",
        # llm_evaluation.txt
        "What is RAGAS and what does it measure?",
        "How does faithfulness differ from answer relevancy?",
        "What is a typical good faithfulness score for a RAG pipeline?",
        "How is hallucination risk typically assessed?",
        "What does context precision evaluate?",
        # cross_encoder_reranking.txt
        "What is cross-encoder reranking?",
        "How does two-stage retrieval work?",
        "Why are cross-encoders more accurate than bi-encoders?",
        "What cross-encoder models are mentioned?",
        "How does two-stage retrieval balance latency and accuracy?",
        # multi_agent_consensus.txt
        "What is multi-agent consensus in RAG?",
        "How does multi-agent consensus improve answer reliability?",
        "How many LLM instances are typically used in consensus?",
        "How is the best answer selected from multiple candidates?",
        "What additional verification step strengthens trust after consensus?",
        "Explain the full RAG retrieval-to-answer pipeline including reranking and consensus.",
        "How does INT4 quantization like GPTQ preserve accuracy compared to naive quantization?",
    ]

print(f"Eval set: {len(EVAL_QUERIES)} queries across {len(SAMPLE_DOCS)} topics")

Eval set: 27 queries across 12 topics


---
## 4. Benchmark Runner

`run_config()` instantiates a fresh pipeline with the given reranker on/off and consensus n, then runs all 25 queries. Results include latency, confidence, chunk counts, and full answer text.

In [8]:
#from multiagent_rag_system.agent.pipeline.pipeline import RAGOrchestrator
from multiagent_rag_system.agent.agents.consensus_agent import ConsensusAgent
from multiagent_rag_system.agent.agents.reranker_agent import RerankerAgent
from multiagent_rag_system.agent.agents.evaluator import RAGASEvaluator
from multiagent_rag_system.agent.agents.confidence_score_agent import ConfidenceScoringAgent
from multiagent_rag_system.agent.agents.claim_verification_agent import ClaimVerificationAgent
from multiagent_rag_system.agent.agents.query_expansion import QueryExpansionAgent
from multiagent_rag_system.agent.agents.retrieval_agent import ChunkRetrieval
from multiagent_rag_system.src.cache.cache import SemanticCache
from multiagent_rag_system.src.models.models import QueryRequest


class NoopCache:
    """Cache that always misses and discards sets — ensures every query runs fresh."""
    async def get(self, key): return None
    async def set(self, key, value): pass


@dataclass
class ConfigResult:
    label: str
    reranker_on: bool
    n_generators: int
    expansion_strategy: str
    expander_on:bool
    results: list = field(default_factory=list)


@dataclass
class QueryResult:
    query: str
    latency_ms: float
    answer: str
    exp_str:str
    exp_on:str
    n_retrieved: int
    n_reranked: int
    reranked_chunk: list
    n_expanded: int
    n_claims: int
    n_supported: int
    claim_support: float
    confidence_final: float
    hallucination_risk: str
    error: Optional[str] = None
    #populated later
    faithfulness: Optional[float] = None
    answer_relevancy: Optional[float] = None
    context_precision: Optional[float] = None
    context_recall: Optional[float] = None
    grounding_score: Optional[float] = None  # LLM-judge, populated later



async def run_config(label: str, reranker_on: bool, 
                     n_generators: int, expander_on:bool,
                     exp_strg:str,
                     ) -> ConfigResult:
    
    """Run all eval queries under a single pipeline config."""
   
   # os.environ["ACTIVE_PROVIDER"]=active_provider
    reranker = RerankerAgent()
    reranker.config.enabled = reranker_on
    expander = QueryExpansionAgent()
    expander.config.enabled = expander_on
    expander.config.strategy = exp_strg
    consensus = ConsensusAgent(n=n_generators)

    pipeline = RAGOrchestrator(
        expansion=expander,
        reranker=reranker,
        consensus=consensus,
        cache=NoopCache(),
    )

    results = []
    n_queries = len(EVAL_QUERIES)
    for i, q_text in enumerate(EVAL_QUERIES):
        print(f"  [{i+1}/{n_queries}] {label}", end="")
        t0 = time.perf_counter()
        try:
            req = QueryRequest(query=q_text, include_trace=False)
            resp = await pipeline.run(req)
            elapsed = (time.perf_counter() - t0) * 1000
  
            n_supported = sum(1 for c in resp.claims if c.supported)
            results.append(QueryResult(
                    query=q_text,
                    latency_ms=resp.latency_ms,
                    answer=resp.answer,
                    exp_str=exp_strg,
                    exp_on=expander_on,
                    n_retrieved=len(resp.retrieved_chunks),
                    n_reranked=len(resp.reranked_chunks),
                    reranked_chunk = resp.reranked_chunks,
                    n_expanded=len(resp.expanded_queries),
                    n_claims=len(resp.claims),
                    n_supported=n_supported,
                    claim_support=resp.confidence.claim_support,
                    confidence_final=resp.confidence.final,
                    hallucination_risk=resp.hallucination_risk.value,
                ))
            print(f"  {resp.latency_ms:7.0f}ms  risk={resp.hallucination_risk.value}")
            time.sleep(30)

        except Exception as e:
            print("==>>>>Generation failed!!!")
            raise e

    return ConfigResult(label=label, reranker_on=reranker_on, n_generators=n_generators, 
                        expansion_strategy=exp_strg, expander_on=expander_on, results=results)

/home/emmanuel/Desktop/Multiagent_RAG_system/.venv/lib/python3.13/site-packages/instructor/client_gemini.py:6: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  import google.generativeai as genai


---
## 5. Ragas Evaluator and LLM-Judge Grounding
- Use Ragas for checking the Faithfulness, Answer Relevancy, Context Precision and Context Recall
- For each answer, ask the LLM to rate factual grounding on a 1-5 scale.

We sample every answer from all configs (up to 400 calls).

In [9]:
get_settings.cache_clear()
%reload_ext autoreload
%autoreload 2 
import importlib
from multiagent_rag_system.agent.agents.evaluator import RAGASEvaluator
async def ragas_eval(data:dict, provider:str):
    from multiagent_rag_system.agent.agents.evaluator import RAGASEvaluator
    evaluator = RAGASEvaluator(provider)
    length = len(data)
    index = 0
    for i, (key, value) in enumerate(data.items()):
        leng = len(value.results)
        for idx, qr in enumerate(value.results):
            if qr.faithfulness is not None:
                index+=1
                continue
                
            eval= await evaluator.evaluate(qr.query, qr.answer, 
                                    qr.reranked_chunk)
            
            qr.faithfulness=round(eval.faithfulness, 4)
            qr.answer_relevancy = round(eval.answer_relevancy,4)
            qr.context_precision = round(eval.context_precision,4)
            qr.context_recall= round(eval.context_recall,4)
            
            index+=1
            print(f">>>> {index}/{length*leng} Ragas Evaluations Completed")
    print("operations completed!")
    #return data

In [10]:
async def grounding_eval(data: dict, provider: str, sleep_secs: float = None):
    """LLM-judge grounding 1-5.

    Fixes vs earlier version:
      - strict standalone integer regex (\b[1-5]\b), no spurious digit pick
      - parse failure stored as None, NOT coerced to the worst 1.0
      - correct skip/accounting for already-graded and failed answers
      - configurable sleep for rate limits (env GROUNDING_SLEEP_SECS)
    """
    from multiagent_rag_system.src.llm.llms import get_llm_client

    GROUNDING_PROMPT = (
        "You are evaluating whether a RAG system's answer is factually grounded "
        "in the retrieved source chunks provided below.\n\n"
        "Sources (retrieved chunks):\n{Sources}\n\n"
        "Question:\n{Question}\n\n"
        "Answer to evaluate:\n{Answer}\n\n"
        "Instructions:\n"
        "1. Break the answer down into its individual factual claims.\n"
        "2. For each claim, check whether it is supported by the Sources - "
        "paraphrasing is fine, but the claim must be entailed by what the "
        "Sources actually say, not just plausible or true in general.\n"
        "3. Do NOT reward or penalize writing quality, fluency, or style - "
        "grade grounding only.\n"
        "4. If the answer explicitly states the Sources do not contain enough "
        "information to answer (i.e., it correctly declines rather than guessing), "
        "treat this as fully grounded (5), not as a failure.\n"
        "5. A claim that is true but not stated or implied by the Sources "
        "counts as unsupported - the answer must be grounded in these "
        "specific Sources, not in general knowledge.\n\n"
        "Rate the answer on a scale of 1 to 5:\n"
        "1 = No claims are supported by the Sources; answer is fabricated or "
        "entirely drawn from outside knowledge.\n"
        "2 = A small minority of claims are supported; most of the answer is "
        "unsupported or fabricated.\n"
        "3 = Roughly half the claims are supported; a meaningful portion of "
        "the answer is unsupported.\n"
        "4 = Nearly all claims are supported, with one minor unsupported "
        "addition or overstatement.\n"
        "5 = Every claim is fully supported by the Sources, OR the answer "
        "correctly declines to answer due to insufficient source coverage.\n\n"
        "Return ONLY a single integer 1-5, nothing else."
    )

    if sleep_secs is None:
        sleep_secs = float(os.environ.get("GROUNDING_SLEEP_SECS", "60"))

    get_llm_client.cache_clear()
    llm = get_llm_client(provider)

    print("Running LLM judge on all answers...")
    total = sum(len(cr.results) for cr in data.values())
    done = 0
    skipped = 0
    failed = 0
    for label, cr in data.items():
        for qr in cr.results:
            if qr.grounding_score:
                skipped += 1
                continue
            done += 1
            try:
                resp = await llm.complete(
                    system=GROUNDING_PROMPT,
                    user=(f"Question: {qr.query}\n\nSources: \n"
                          f"{[c.chunk.content for c in qr.reranked_chunk]}\n\n"
                          f"Answer: {qr.answer}"),
                    temperature=0.0,
                )
                import re
                match = re.search(r'\b([1-5])\b', resp.text.strip())
                qr.grounding_score = float(match.group(1)) if match else None
                print(f"  ==>>> {label}:{qr.query} --> grounding score: {qr.grounding_score} ")

                if qr.grounding_score is None:
                    failed += 1
                    print(f"[warn] no 1-5 grade parsed -> None for {qr.query!r}: "
                          f"{resp.text.strip()[:80]!r}")
                if sleep_secs:
                    time.sleep(sleep_secs)
            except Exception as e:
                qr.grounding_score = None
                failed += 1
                print(f"[warn] grounding eval failed for {qr.query!r}: {e}")
            if done % 10 == 0:
                print(f"  ... judged {done}/{total} (failed={failed}, skipped={skipped})")
    print(f"Done. Newly judged {done}, skipped {skipped}, failed {failed}.")
    #return data

---
## 6. Run All Configs

4 configurations: A (ON, 3gen), B (ON, 1gen), C (OFF, 3gen), D (OFF, 1gen).

Each runs all 25 queries through a fresh pipeline.

In [ ]:
CONFIGS = [
    ("A: Reranker=ON, 3gen, exp_str=both, exp_on=ON",         True,  3, 'both',        True),
    ("B: Reranker=ON, 3gen, exp_str=hyde, exp_on=ON",         True,  3, 'hyde',        True),
    ("C: Reranker=ON, 3gen, exp_str=multi_query, exp_on=ON",  True,  3, 'multi_query', True),
    ("D: Reranker=ON, 3gen, exp_str=off, exp_on=OFF",         True,  3, 'off',         False),
    ("E: Reranker=ON, 1gen, exp_str=both, exp_on=ON",         True,  1, 'both',        True),
    ("F: Reranker=ON, 1gen, exp_str=hyde, exp_on=ON",         True,  1, 'hyde',        True),
    ("G: Reranker=ON, 1gen, exp_str=multi_query, exp_on=ON",  True,  1, 'multi_query', True),
    ("H: Reranker=ON, 1gen, exp_str=off, exp_on=OFF",         True,  1, 'off',         False),
    ("I: Reranker=OFF, 3gen, exp_str=both, exp_on=ON",        False, 3, 'both',        True), 
    ("J: Reranker=OFF, 3gen, exp_str=hyde, exp_on=ON",        False, 3, 'hyde',        True),
    ("K: Reranker=OFF, 3gen, exp_str=multi_query, exp_on=ON", False, 3, 'multi_query', True),
    ("L: Reranker=OFF, 3gen, exp_str=off, exp_on=OFF",        False, 3, 'off',         False),
    ("M: Reranker=OFF, 1gen, exp_str=both, exp_on=ON",        False, 1, 'both',        True),
    ("N: Reranker=OFF, 1gen, exp_str=hyde, exp_on=ON",        False, 1, 'hyde',        True),
    ("O: Reranker=OFF, 1gen, exp_str=multi_query, exp_on=ON", False, 1, 'multi_query', True),
    ("P: Reranker=OFF, 1gen, exp_str=off, exp_on=OFF",        False, 1, 'off',         False),
]

all_results: dict[str, ConfigResult] = {}
track:list[str] = ['A: Reranker=ON, 3gen, exp_str=both, exp_on=ON', 
                   'B: Reranker=ON, 3gen, exp_str=hyde, exp_on=ON', 
                   'C: Reranker=ON, 3gen, exp_str=multi_query, exp_on=ON',
                   "D: Reranker=ON, 3gen, exp_str=off, exp_on=OFF",
                   "E: Reranker=ON, 1gen, exp_str=both, exp_on=ON",
                   "F: Reranker=ON, 1gen, exp_str=hyde, exp_on=ON",
                   'G: Reranker=ON, 1gen, exp_str=multi_query, exp_on=ON', 
                   'H: Reranker=ON, 1gen, exp_str=off, exp_on=OFF',
                   'I: Reranker=OFF, 3gen, exp_str=both, exp_on=ON',
                   "J: Reranker=OFF, 3gen, exp_str=hyde, exp_on=ON",
                   "K: Reranker=OFF, 3gen, exp_str=multi_query, exp_on=ON",
                   "L: Reranker=OFF, 3gen, exp_str=off, exp_on=OFF", 
                   "M: Reranker=OFF, 1gen, exp_str=both, exp_on=ON",
                   "N: Reranker=OFF, 1gen, exp_str=hyde, exp_on=ON",
                   "O: Reranker=OFF, 1gen, exp_str=multi_query, exp_on=ON",
                   "P: Reranker=OFF, 1gen, exp_str=off, exp_on=OFF"
                   ]
for label, reranker_on, n_gen, exp_str, exp_on in CONFIGS:
    print(f"\n{'='*60}")
    print(f"Running {label}")
    print(f"{'='*60}")
    if label in track:
        continue
    cr = await run_config(label, reranker_on, n_gen, exp_on, exp_str,)
    all_results[label] = cr
    track.append(label)

In [ ]:
get_settings.cache_clear()
%reload_ext autoreload
%autoreload 2
await ragas_eval(all_results, provider='gemini')

No device provided, using cpu
Loading SentenceTransformer model from sentence-transformers/all-MiniLM-L6-v2.
HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/openai/chat/completions "HTTP/1.1 200 OK"
HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/openai/chat/completions "HTTP/1.1 200 OK"
HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/openai/chat/completions "HTTP/1.1 200 OK"
HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/openai/chat/completions "HTTP/1.1 200 OK"
HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/openai/chat/completions "HTTP/1.1 200 OK"
Retrying request to /chat/completions in 0.420428 seconds
HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/openai/chat/completions "HTTP/1.1 200 OK"
HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/openai/chat/completions "HTTP/1.1 200 OK"
Retrying request to /chat/completions in 0.806012 seconds
HTTP 

>>>> 186/189 Ragas Evaluations Completed


HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/openai/chat/completions "HTTP/1.1 200 OK"
HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/openai/chat/completions "HTTP/1.1 200 OK"
HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/openai/chat/completions "HTTP/1.1 200 OK"
HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/openai/chat/completions "HTTP/1.1 200 OK"
HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/openai/chat/completions "HTTP/1.1 200 OK"
HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/openai/chat/completions "HTTP/1.1 200 OK"
HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/openai/chat/completions "HTTP/1.1 200 OK"
Retrying request to /chat/completions in 0.479303 seconds
HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/openai/chat/completions "HTTP/1.1 200 OK"
HTTP Request: POST https://generativelanguage.googleapis.com/v

>>>> 187/189 Ragas Evaluations Completed


HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/openai/chat/completions "HTTP/1.1 200 OK"
HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/openai/chat/completions "HTTP/1.1 200 OK"
HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/openai/chat/completions "HTTP/1.1 200 OK"
Retrying request to /chat/completions in 0.485999 seconds
HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/openai/chat/completions "HTTP/1.1 200 OK"
Retrying request to /chat/completions in 0.459348 seconds
HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/openai/chat/completions "HTTP/1.1 200 OK"
HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/openai/chat/completions "HTTP/1.1 200 OK"
HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/openai/chat/completions "HTTP/1.1 200 OK"
HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/openai/chat/completions "HTTP/1.1 200 OK"
Retr

>>>> 188/189 Ragas Evaluations Completed


HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/openai/chat/completions "HTTP/1.1 200 OK"
HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/openai/chat/completions "HTTP/1.1 200 OK"
HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/openai/chat/completions "HTTP/1.1 200 OK"
HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/openai/chat/completions "HTTP/1.1 200 OK"
HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/openai/chat/completions "HTTP/1.1 200 OK"
HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/openai/chat/completions "HTTP/1.1 200 OK"
HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/openai/chat/completions "HTTP/1.1 200 OK"
HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/openai/chat/completions "HTTP/1.1 200 OK"
HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/openai/chat/completions "HTTP/1.1 200 OK"
Batches: 1

>>>> 189/189 Ragas Evaluations Completed
operations completed!


In [ ]:
await grounding_eval(all_results, provider='groq', sleep_secs=20)

### Raw Data Export

In [ ]:
# Export all raw results as JSON for external analysis
from dataclasses import asdict, is_dataclass

def json_default(o):
    if is_dataclass(o):
        return asdict(o)
    if hasattr(o, "__dict__"):
        return o.__dict__
    return str(o)  # fallback for anything else (e.g. numpy types, enums)

def qr_to_dict(qr: QueryResult) -> dict:
    return dict(
        query=qr.query,
        latency_ms=qr.latency_ms,
        answer=qr.answer,
        expander_strategy=qr.exp_str,
        expander_on=qr.exp_on,
        n_retrieved=qr.n_retrieved,
        n_reranked=qr.n_reranked,
        n_expanded=qr.n_expanded,
        n_claims=qr.n_claims,
        reranked_chunks = json_default(qr.reranked_chunk),
        n_supported=qr.n_supported,
        claim_support=qr.claim_support,
        confidence_final=qr.confidence_final,
        hallucination_risk=qr.hallucination_risk,
        error=qr.error,
        grounding_score=qr.grounding_score,
        faithfulness= qr.faithfulness,
        answer_relevancy = qr.answer_relevancy,
        context_precision=qr.context_precision,
        context_recall = qr.context_recall
    )

export = {
    label: {
        "config": {"reranker_on": cr.reranker_on, "n_generators": cr.n_generators,
                   "expander_on": cr.expander_on, "expander_strategy": cr.expansion_strategy},
        "results": [qr_to_dict(qr) for qr in cr.results],
    }
    for label, cr in all_results.items()
}

export_path = REPO_ROOT / "notebook" / "benchmark_raw.json"
with open(export_path, "a") as f:
    json.dump(export, f, indent=2)
print(f"Raw results exported to {export_path}")

Raw results exported to /home/emmanuel/Desktop/Multiagent_RAG_system/notebook/benchmark_raw.json
